## Prerequisites

This notebook demonstrates how to invoke a **deployed real-time inference endpoint** on Azure for the Clinical De-identification model. Follow the steps below in order before running this notebook.

### Deployment Sequence

1. **Acquire the Azure Application Offer** — Subscribe to the *clinical_deidentification_docwise_wip_en* offer in the [Azure Marketplace](https://marketplace.azure.com) to obtain the Solution Template.
2. **Launch guided deployment** — From the Marketplace listing, start the guided deployment flow for the solution.
3. **Supply configuration** — Complete the guided UX forms (defined by the offer's `createUiDefinition`) with the required parameters.
4. **Deploy resources** — Confirm deployment; Azure provisions the resources defined in the offer's `mainTemplate.json`.
5. **Wait for deployment to succeed** — Verify the deployment reaches a *Succeeded* state before proceeding. Note the endpoint name from the deployment output.
6. **Run this notebook** — Execute the cells below against the real-time inference endpoint provisioned by the deployment.

### Environment

- **Notebook runtime**: This notebook is intended to run in **Azure Machine Learning workspace notebooks** (e.g. via an Azure ML Studio compute instance or Studio notebooks).
- **Jupyter kernel**: Use a kernel with the `azure.ai.ml` Python package installed, for example **Python 3.10 — SDK v2** (`python310-sdkv2`).
- **Authentication**: The kernel must be authenticated to Azure so that Azure ML SDK calls succeed (e.g. `DefaultAzureCredential()`). The authenticated identity must be granted Azure RBAC permissions appropriate for reading the workspace and online endpoint configuration, and for retrieving endpoint credentials. Consult your organization's access-policy guidance and the [Azure ML endpoint credential documentation](https://learn.microsoft.com/azure/machine-learning/how-to-manage-endpoints) for details.
- **Network access**: The notebook's compute instance must be able to reach the endpoint's `scoring_uri` (no outbound firewall or VNET restrictions that block it).


## Deidentify Clinical Document (EN)

- **Model**: [clinical_deidentification_docwise_wip_en](https://nlp.johnsnowlabs.com/2024/10/03/clinical_deidentification_docwise_wip_en.html)

- **Model Description**: This pipeline de-identifies protected health information (PHI) from clinical and medical text by detecting sensitive entities and replacing them with placeholder tags (masked) or plausible surrogate values (obfuscated). 
The pipeline can mask and obfuscate `LOCATION`, `CONTACT`, `PROFESSION`, `NAME`, `DATE`, `ID`, `AGE`, `MEDICALRECORD`, `ORGANIZATION`, `HEALTHPLAN`, `DOCTOR`, `USERNAME`,
`LOCATION-OTHER`, `URL`, `DEVICE`, `CITY`, `ZIP`, `STATE`, `PATIENT`, `COUNTRY`, `STREET`, `PHONE`, `HOSPITAL`, `EMAIL`, `IDNUM`, `BIOID`, `FAX`, `LOCATION_OTHER`, `DLN`,
`SSN`, `ACCOUNT`, `PLATE`, `VIN`, `LICENSE`, `IP` entities.

In [ ]:
ENDPOINT_NAME = "<CUSTOMER_TO_PUT_DEPLOYED_ENDPOINT_NAME>"

### Input Format

To use the model, you need to provide input in one of the following supported formats:

#### JSON Format

Provide input as JSON. We support two variations within this format:

1. **Array of Text Documents**: 
   Use an array containing multiple text documents. Each element represents a separate text document.

   ```json
   {
       "text": [
           "Text document 1",
           "Text document 2",
           ...
       ]
   }

    ```

2. **Single Text Document**:
   Provide a single text document as a string.


   ```json
    {
        "text": "Single text document"
    }
   ```

#### JSON Lines (JSONL) Format

Provide input in JSON Lines format, where each line is a JSON object representing a text document.

```
{"text": "Text document 1"}
{"text": "Text document 2"}
```

### Important Parameter

- **masking_policy**: `str`

    Users can select a masking policy to determine how sensitive entities are handled:

    - **masked**: Default policy that masks entities with their type.

      Example: "My name is Mike. I was admitted to the hospital yesterday."  
      -> "My name is `<PATIENT>`. I was admitted to the hospital yesterday."

    - **obfuscated**: Replaces sensitive entities with random values of the same type.

      Example: "My name is Mike. I was admitted to the hospital yesterday."  
      -> "My name is `Barbaraann Share`. I was admitted to the hospital yesterday."

    
You can specify these parameters in the input as follows:

```json
{
    "text": [
        "Text document 1",
        "Text document 2"
    ],
    "masking_policy": "masked"
}
```

> **Note:** Demonstration inputs used with this endpoint should consist of approved synthetic or properly de-identified data. Never submit real protected health information (PHI) without explicit authorization and the required safeguarding controls.


### Authentication and Endpoint Lookup

The cell below uses `DefaultAzureCredential()` to authenticate to Azure, then retrieves the endpoint's scoring URI and primary access key from the workspace configuration. If you see authentication errors, verify that you are logged into Azure Cloud Shell / your compute instance and that your identity has sufficient role assignments on the workspace.


In [35]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
import requests
import json


ml_client = MLClient.from_config(DefaultAzureCredential())
endpoint_url = ml_client.online_endpoints.get(ENDPOINT_NAME).scoring_uri
key = ml_client.online_endpoints.get_keys(ENDPOINT_NAME).primary_key


def invoke_realtime_endpoint(record, content_type="application/json", accept="application/json"):
    response = requests.post(
        endpoint_url,
        headers={
            "Authorization": f"Bearer {key}",
            "Content-Type": content_type,
        },
        data=json.dumps(record) if content_type == "application/json" else record,
     
    )

    response_body = response.content.decode("utf-8")

    if accept == "application/json":
        return json.loads(response_body)
    elif accept == "application/jsonlines":
        return response_body
    else:
        raise ValueError(f"Unsupported accept type: {accept}")

Found the config file in: /config.json
Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


### Sample Data (Synthetic — for demonstration only)

The following cells define sample clinical text documents used in the examples below. When using this notebook with your own data, ensure all inputs are approved synthetic or properly de-identified. Never submit real protected health information (PHI) without explicit authorization and appropriate controls.


In [36]:
docs = [
'''Mr. William Garcia, 25 years-old , born in Beijing, was transfered to the The Johns Hopkins Hospital. Phone number: (541) 754-3010. MSW: 100009632582 . for his colonic polyps.  William Garcia wants to know the results from them.  He is not taking hydrochlorothiazide and is curious about his blood pressure. He said he has cut his alcohol back to 6 pack once a week.  William Garcia has cut back his cigarettes to one time per week. MR. # 50712249 P:   Follow up with Dr. Hobbs Spruce in 3 months. Gilbert P. Perez, M.D. Johns Hopkins Hospital 1800 Orleans St. PHONE: (+1) 410-955-5000 ''',

'''Record date : 2022-01-13 , Ethan Hale , M.D . , Name : Ava Davis. EMAIL: davis_a@gmail.com. MR # 7190334  Phone : (555) 555-1212. Date : 01-13-1993 PCP : Oliveira , 29 years-old. A long history of insulin dependent diabetes, type 2; coronary artery disease; chronic renal insufficiency; peripheral vascular disease, also secondary to diabetes; who was originally admitted to an outside hospital for what appeared to be acute paraplegia, lower extremities.  Ava Davis was noted to have a large sacral wound; this is in a similar location with his previous laminectomy, and this continues to receive daily care. DD: 01/13/2022 . DV: 01/13/2022. Ethan Hale , M.D. MSW 754443200936. NewYork-Presbyterian Hospital . 525 East 68th Street. (85) 555-1212. ''',

]

sample_text = """Record date : 2013-01-13, David Hale, M.D. is manager,  Name: Elvis Presley. Age: 17. Phone: (9) 7765-5632. MR. # 7194334 Date: 01-13-1993 PCP: Oliveira. Record date: 2012-11-09. Cocke County Baptist Hospital 0295 Keats Street. This 17-yr-old male, presented with chest heaviness that started during a pick-up basketball game. His past medical history was unremarkable. Elvis Presley denied prior cardiac symptoms and suffered no chest trauma during the game. His father had suffered an acute myocardial infarction at age 38. Elvis Presley was a nonsmoker, did not drink alcohol, and denied recreational drug use. Elvis Presley swallowed a tablet of aspirin before coming to the emergency room. His blood pressure was 160/90 mm Hg, and his heart rate was 80 bpm. Physical examination revealed no stigmata of Marfan syndrome. The rest of his physical examination was normal. DD: 01/13/2013 . DV: 01/13/2013, Cocke County Baptist Hospital 0295 Keats Street. PHONE : (+1) 423-625-2200 """

### JSON

#### Example 1: masked (default-policy)

In [37]:
input_json_data = {"text": sample_text}
response_json = invoke_realtime_endpoint(input_json_data, content_type="application/json", accept="application/json")
response_json

{'predictions': 'Record date : <DATE>, <DOCTOR>, M.D. is <PROFESSION>,  Name: <PATIENT>. Age: <AGE>. Phone: (<AGE>) <AGE>5-5632. MR. # <MEDICALRECORD> Date: <DATE> PCP: <DOCTOR>. Record date: <DATE>. <HOSPITAL> <STREET>. This <AGE>-yr-old male, presented with chest heaviness that started during a <PROFESSION>. His past medical history was unremarkable. <PATIENT> denied prior cardiac symptoms and suffered no chest trauma during the game. His father had suffered an acute myocardial infarction at age <AGE>. Elvis <PATIENT> was a nonsmoker, did not drink alcohol, and denied recreational drug use. <PATIENT> swallowed a tablet of aspirin before coming to the emergency room. His blood pressure was 160/90 mm Hg, and his heart rate was 80 bpm. Physical examination revealed no stigmata of Marfan syndrome. The rest of his physical examination was normal. DD: <DATE> . DV: <DATE>, Cocke <LOCATION> <STREET>. PHONE : <PHONE>'}

#### Example 2: obfuscated

In [38]:
input_json_data = {"text": sample_text, "masking_policy": "obfuscated"}
response_json = invoke_realtime_endpoint(input_json_data, content_type="application/json", accept="application/json")
response_json

{'predictions': 'Record date : 2013-03-10, Monroe Hoot, M.D. is dentist,  Name: Ballard Kemps. Age: 18. Phone: (8) 7765-5632. MR. # 6827447 Date: 03-10-1993 PCP: Rosalia. Record date: 2013-01-04. Shawnee Mission Prairie Star 1540 Maple Rd. This 18-yr-old male, presented with chest heaviness that started during a psychiatrist. His past medical history was unremarkable. Ballard Kemps denied prior cardiac symptoms and suffered no chest trauma during the game. His father had suffered an acute myocardial infarction at age 32. Elvis Kemps was a nonsmoker, did not drink alcohol, and denied recreational drug use. Ballard Kemps swallowed a tablet of aspirin before coming to the emergency room. His blood pressure was 160/90 mm Hg, and his heart rate was 80 bpm. Physical examination revealed no stigmata of Marfan syndrome. The rest of his physical examination was normal. DD: 03/10/2013 . DV: 03/10/2013, Cocke 1409 9th Street 1540 Maple Rd. PHONE : (+8) 714-310-1155'}

### JSON Lines

In [39]:
def create_jsonl(records, masking_policy=None):
    json_records = []

    if isinstance(records, str):
        records = [records]

    for text in records:
        record = {"text": text}

        if masking_policy is not None:
            record["masking_policy"] = masking_policy
        json_records.append(record)

    json_lines = '\n'.join(json.dumps(record) for record in json_records)
    return json_lines


#### Example 1: masked (default-policy)

In [40]:
input_jsonl_data = create_jsonl(docs, masking_policy="masked")
data = invoke_realtime_endpoint(input_jsonl_data, content_type="application/jsonlines" , accept="application/jsonlines" )
print(data)

{"predictions": "Mr. <PATIENT>, <AGE> years-old , born in <CITY>, was transfered to the The <HOSPITAL>. Phone number: <PHONE>. MSW: <PHONE> . for his colonic polyps.  <PATIENT> wants to know the results from them.  He is not taking hydrochlorothiazide and is curious about his blood pressure. He said he has cut his alcohol back to 6 pack once a week.  <PATIENT> has cut back his cigarettes to one time per week. MR. # <IDNUM> P:   Follow up with Dr. <CITY> <CITY> in 3 months. <DOCTOR> P. <DOCTOR>, M.D. <HOSPITAL> <STREET>. PHONE: <PHONE>"}
{"predictions": "Record date : <DATE> , <DOCTOR> , M.D . , Name : <DOCTOR>. <NAME>: <EMAIL>. MR # <MEDICALRECORD>  Phone : <PHONE>. Date : <DATE> PCP : <DOCTOR> , <AGE> years-old. A long history of insulin dependent diabetes, type 2; coronary artery disease; chronic renal insufficiency; peripheral vascular disease, also secondary to diabetes; who was originally admitted to an outside hospital for what appeared to be acute paraplegia, lower extremities. 

#### Example 2: obfuscated

In [41]:
input_jsonl_data = create_jsonl(docs, masking_policy="obfuscated")
data = invoke_realtime_endpoint(input_jsonl_data, content_type="application/jsonlines" , accept="application/jsonlines" )
print(data)

{"predictions": "Mr. Salena Meals, 39 years-old , born in Olathe, was transfered to the The Northside Medical Center. Phone number: (078) 607-4585. MSW: 855552341091 . for his colonic polyps.  Salena Meals wants to know the results from them.  He is not taking hydrochlorothiazide and is curious about his blood pressure. He said he has cut his alcohol back to 6 pack once a week.  Salena Meals has cut back his cigarettes to one time per week. MR. # 05681172 P:   Follow up with Dr. Avoca Magee in 3 months. Shellie P. Consuela, M.D. Northside Medical Center 1118 S Farmerville St. PHONE: (+8) 785-200-0555"}
{"predictions": "Record date : 2022-03-10 , Rogerio Hoot , M.D . , Name : Pasquale Grit. DARLYNE: felicitas@yahoo.com. MR # 6825447  Phone : (000) 000-8181. Date : 03-10-1993 PCP : Rosalia , 23 years-old. A long history of insulin dependent diabetes, type 2; coronary artery disease; chronic renal insufficiency; peripheral vascular disease, also secondary to diabetes; who was originally a